In [1]:
!pip install -q pandas statsmodels
print("✅ Dependencies ready")

✅ Dependencies ready


In [2]:
import pandas as pd
from statsmodels.tsa.stattools import adfuller
print("✅ Imports done")

✅ Imports done


In [3]:
# ──────────────────────────────────────────────────────────────────
# 📂 INSTRUCTIONS:
#    1. Click the 📁 folder icon in the LEFT sidebar of Colab
#    2. Drag & drop your 4 CSV files directly into that panel
#    3. They will appear at /content/FILENAME.csv
#    4. Then run this cell
# ──────────────────────────────────────────────────────────────────

import os

FILE_PATHS = {
    "FEDFUNDS"    : "/content/FEDFUNDS.csv",
    "MORTGAGE30US": "/content/MORTGAGE30US.csv",
    "PCE"         : "/content/PCE.csv",
    "UNRATE"      : "/content/UNRATE.csv",
}

# Verify all files are present
all_found = True
for name, path in FILE_PATHS.items():
    found = os.path.exists(path)
    status = "✅ Found" if found else "❌ NOT FOUND"
    print(f"  {status}  →  {path}")
    if not found:
        all_found = False

if all_found:
    print("\n✅ All files found — ready to load!")
else:
    print("\n⚠️  Some files are missing. Drag & drop them into the Colab file panel first.")

  ✅ Found  →  /content/FEDFUNDS.csv
  ✅ Found  →  /content/MORTGAGE30US.csv
  ✅ Found  →  /content/PCE.csv
  ✅ Found  →  /content/UNRATE.csv

✅ All files found — ready to load!


In [4]:
def load_fred(filename):
    return (
        pd.read_csv(filename, parse_dates=["observation_date"])
        .set_index("observation_date")
        .replace(".", pd.NA)
        .apply(pd.to_numeric, errors="coerce")
    )

fedfunds = load_fred(FILE_PATHS["FEDFUNDS"])
mortgage = load_fred(FILE_PATHS["MORTGAGE30US"])
pce      = load_fred(FILE_PATHS["PCE"])
unrate   = load_fred(FILE_PATHS["UNRATE"])

print("✅ All files loaded")
print(f"   FEDFUNDS     : {fedfunds.shape[0]} rows  |  {fedfunds.index.min().date()} → {fedfunds.index.max().date()}")
print(f"   MORTGAGE30US : {mortgage.shape[0]} rows  |  {mortgage.index.min().date()} → {mortgage.index.max().date()}")
print(f"   PCE          : {pce.shape[0]} rows  |  {pce.index.min().date()} → {pce.index.max().date()}")
print(f"   UNRATE       : {unrate.shape[0]} rows  |  {unrate.index.min().date()} → {unrate.index.max().date()}")

✅ All files loaded
   FEDFUNDS     : 132 rows  |  2015-01-01 → 2025-12-01
   MORTGAGE30US : 574 rows  |  2015-01-08 → 2025-12-31
   PCE          : 132 rows  |  2015-01-01 → 2025-12-01
   UNRATE       : 132 rows  |  2015-01-01 → 2025-12-01


In [5]:
mortgage_monthly = mortgage.resample("MS").mean()

print(f"✅ MORTGAGE30US resampled to monthly")
print(f"   Rows: {len(mortgage)} (weekly) → {len(mortgage_monthly)} (monthly)")
mortgage_monthly.head()

✅ MORTGAGE30US resampled to monthly
   Rows: 574 (weekly) → 132 (monthly)


,MORTGAGE30US
observation_date,
2015-01-01,3.670
2015-02-01,3.710
2015-03-01,3.770
2015-04-01,3.672
2015-05-01,3.840


In [6]:
df = fedfunds.join([mortgage_monthly, pce, unrate], how="outer").sort_index()

print(f"✅ Merged DataFrame: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"   Columns    : {list(df.columns)}")
print(f"   Date range : {df.index.min().date()} → {df.index.max().date()}")
df.head()

✅ Merged DataFrame: 132 rows × 4 columns
   Columns    : ['FEDFUNDS', 'MORTGAGE30US', 'PCE', 'UNRATE']
   Date range : 2015-01-01 → 2025-12-01


,FEDFUNDS,MORTGAGE30US,PCE,UNRATE
observation_date,,,,
2015-01-01,0.11,3.670,12066.7,5.7
2015-02-01,0.11,3.710,12116.6,5.5
2015-03-01,0.11,3.770,12176.1,5.4
2015-04-01,0.12,3.672,12209.1,5.4
2015-05-01,0.12,3.840,12275.4,5.6


In [7]:
# PCE → Year-over-Year % change (inflation proxy)
df["PCE_YOY"] = df["PCE"].pct_change(12) * 100
df.drop(columns=["PCE"], inplace=True)

# Mortgage Spread = Mortgage Rate − Fed Funds Rate
df["MORTGAGE_SPREAD"] = df["MORTGAGE30US"] - df["FEDFUNDS"]
df.drop(columns=["MORTGAGE30US"], inplace=True)

# Drop rows with any NaN
rows_before = len(df)
df.dropna(inplace=True)
rows_after  = len(df)

print(f"✅ Features engineered")
print(f"   PCE       → PCE_YOY (12-month % change)")
print(f"   MORTGAGE30US + FEDFUNDS → MORTGAGE_SPREAD")
print(f"   Dropped NaN rows: {rows_before - rows_after}  |  Remaining: {rows_after}")
df.head()

✅ Features engineered
   PCE       → PCE_YOY (12-month % change)
   MORTGAGE30US + FEDFUNDS → MORTGAGE_SPREAD
   Dropped NaN rows: 13  |  Remaining: 119


,FEDFUNDS,UNRATE,PCE_YOY,MORTGAGE_SPREAD
observation_date,,,,
2016-01-01,0.34,4.8,3.408554,3.5325
2016-02-01,0.38,4.9,3.614050,3.2800
2016-03-01,0.36,5.0,2.964003,3.3340
2016-04-01,0.37,5.1,3.295902,3.2350
2016-05-01,0.37,4.8,3.073627,3.2300


In [8]:
print("=" * 55)
print("ADF Stationarity Test (H0: series has a unit root)")
print("=" * 55)

for col in df.columns:
    result = adfuller(df[col].dropna())
    pval   = result[1]
    status = "✅ Stationary" if pval < 0.05 else "⚠️  Non-stationary"
    print(f"{col:<22} p-value: {pval:.4f}   {status}")

print("=" * 55)

ADF Stationarity Test (H0: series has a unit root)
FEDFUNDS               p-value: 0.3557   ⚠️  Non-stationary
UNRATE                 p-value: 0.0134   ✅ Stationary
PCE_YOY                p-value: 0.2125   ⚠️  Non-stationary
MORTGAGE_SPREAD        p-value: 0.0712   ⚠️  Non-stationary


In [9]:
OUTPUT_CSV = "/content/macro_regimes_ready.csv"
df.to_csv(OUTPUT_CSV)

print(f"✅ File saved: {OUTPUT_CSV}")
print(f"   Shape      : {df.shape[0]} rows × {df.shape[1]} columns")
print(f"   Columns    : {list(df.columns)}")
print(f"   Date range : {df.index.min().date()} → {df.index.max().date()}")
df.head()

✅ File saved: /content/macro_regimes_ready.csv
   Shape      : 119 rows × 4 columns
   Columns    : ['FEDFUNDS', 'UNRATE', 'PCE_YOY', 'MORTGAGE_SPREAD']
   Date range : 2016-01-01 → 2025-12-01


,FEDFUNDS,UNRATE,PCE_YOY,MORTGAGE_SPREAD
observation_date,,,,
2016-01-01,0.34,4.8,3.408554,3.5325
2016-02-01,0.38,4.9,3.614050,3.2800
2016-03-01,0.36,5.0,2.964003,3.3340
2016-04-01,0.37,5.1,3.295902,3.2350
2016-05-01,0.37,4.8,3.073627,3.2300


In [10]:
from google.colab import files
files.download("/content/macro_regimes_ready.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>